<a href="https://colab.research.google.com/github/JakobSchauser/3dTrackingNuclei/blob/main/Ultrack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install ultrack -q
# !pip install tifffile -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.5/266.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.0/128.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 8

In [2]:
import tifffile
import numpy as np
from ultrack import MainConfig, load_config, Tracker, track, to_tracks_layer, tracks_to_zarr
from ultrack.imgproc import robust_invert, detect_foreground
from ultrack.utils.array import array_apply, create_zarr

In [3]:
# load in image
p = "drive/MyDrive/Mattheo/Embryo_37_intrareg_fuse_t0"
paths = ["75", "76"]#, "77", "78"]

imgs = []
for path in paths:
  image = tifffile.imread(p + path + ".tif")
  imgs.append(image[::4,::4, ::4])

In [4]:
images = np.array(imgs)

In [5]:
detection = create_zarr(images.shape, bool, store_or_path="detection.zarr", overwrite=True)
array_apply(
    images,
    out_array=detection,
    func=detect_foreground,
    sigma=25.0,
    voxel_size=[1,1,1],
)




Applying detect_foreground ...: 100%|██████████| 2/2 [00:20<00:00, 10.13s/it]


<Array file://detection.zarr shape=(2, 198, 502, 212) dtype=bool>

In [6]:
boundaries = create_zarr(images.shape, np.float16, store_or_path="boundaries.zarr", overwrite=True)
array_apply(
    images,
    out_array=boundaries,
    func=robust_invert,
    voxel_size=[1,1,1],
)

Applying robust_invert ...: 100%|██████████| 2/2 [00:06<00:00,  3.37s/it]


<Array file://boundaries.zarr shape=(2, 198, 502, 212) dtype=float16>

In [7]:
cfg =  MainConfig()  # or load default config
cfg.segmentation_config.threshold = 0.5
# import prettyprint
from rich.pretty import pprint
# pprint(cfg)

track(
    cfg,
    foreground=detection,
    edges=boundaries,
    scale=[1,1,1],
    overwrite=True,
)


Linking nodes.: 100%|██████████| 1/1 [00:07<00:00,  7.56s/it]


Using Coin-OR Branch and Cut solver
Solving ILP batch 0
Constructing ILP ...
Solving ILP ...
Saving solution ...
Done!
